# 01 - Aggregate the Prescription Cost Analysis monthly files

Runs in Google Colab. Reads the NHS Business Services Authority Prescription Cost Analysis (PCA) monthly files for January 2021 to December 2025 (`pca_YYYYMM.csv`, about 15 GB in total) from Google Drive and aggregates them to the two files the analysis starts from:

- `icb_area_substance_items_full.csv`: items by area code, area name, BNF chemical substance and BNF section;
- `icb_substance_items_by_year.csv`: items by area code, BNF chemical substance and calendar year.

The monthly files are open data: https://opendata.nhsbsa.net/dataset/prescription-cost-analysis-pca-monthly-data (Open Government Licence v3.0). Download them into any folder under My Drive (or add a shortcut to a shared folder there) and run every cell. `DRIVE_DIR` sets where the aggregates are saved.

In [ ]:
# 1. Mount Google Drive and look for the monthly files anywhere under My Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive'   # where the aggregates are saved
import os, glob, re
PATHS = sorted(p for p in glob.glob(f'{DRIVE_DIR}/**/pca_*.csv', recursive=True)
               if re.fullmatch(r'pca_\d{6}\.csv', os.path.basename(p)))
seen, SOURCES = set(), []
for p in PATHS:
    b = os.path.basename(p)
    if b in seen: continue
    seen.add(b); SOURCES.append({'name': b, 'path': p})
SOURCES = [s for s in SOURCES if '202101' <= s['name'][4:10] <= '202512']   # January 2021 to December 2025
print(len(SOURCES), 'monthly files for 2021 to 2025 found under My Drive (deduplicated by name)')

### 2. If none were found: find files shared with you through the Drive API

In [ ]:
if not SOURCES:
    from google.colab import auth; auth.authenticate_user()
    from googleapiclient.discovery import build
    drive_api = build('drive', 'v3')
    found, tok = [], None
    while True:
        r = drive_api.files().list(
            q="name contains 'pca_' and mimeType='text/csv' and trashed=false",
            fields='nextPageToken, files(id,name,size)', pageSize=200, pageToken=tok,
            includeItemsFromAllDrives=True, supportsAllDrives=True).execute()
        found += r.get('files', []); tok = r.get('nextPageToken')
        if not tok: break
    uniq = {f['name']: f for f in found if re.fullmatch(r'pca_\d{6}\.csv', f['name'])}
    SOURCES = [{'name': f['name'], 'id': f['id']} for f in sorted(uniq.values(), key=lambda f: f['name'])
               if '202101' <= f['name'][4:10] <= '202512']
    print('via Drive API:', len(SOURCES), 'files')
else:
    drive_api = None

assert len(SOURCES) == 60, f'Expected the 60 monthly files for January 2021 to December 2025, found {len(SOURCES)}. Put the files (or a shortcut to their folder) under My Drive, then rerun.'
print('total files to process:', len(SOURCES))

### 3. Aggregate one month at a time
Column names differ between the earlier files (Sustainability and Transformation Partnership, STP, fields) and the later ones (ICB fields) and are detected automatically; each printed line shows the columns used. A few minutes when the files are on My Drive, 10 to 20 minutes through the API.

In [ ]:
import os, pandas as pd

def local_path(src):
    if 'path' in src: return src['path'], False
    from googleapiclient.http import MediaIoBaseDownload
    with open('tmp.csv', 'wb') as fh:
        dl = MediaIoBaseDownload(fh, drive_api.files().get_media(fileId=src['id'], supportsAllDrives=True), chunksize=64*1024*1024)
        done = False
        while not done: _, done = dl.next_chunk()
    return 'tmp.csv', True

def pick(cols, *names):
    for n in names:
        if n in cols: return n
    return None

full_parts, year_parts = [], []
for i, src in enumerate(SOURCES, 1):
    path, tmp = local_path(src)
    cols = pd.read_csv(path, nrows=0).columns
    code_c  = pick(cols, 'ICB_CODE', 'STP_CODE', 'SUB_ICB_LOCATION_CODE')
    name_c  = pick(cols, 'ICB_NAME', 'STP_NAME', 'SUB_ICB_LOCATION_NAME')
    drug_c  = pick(cols, 'BNF_CHEMICAL_SUBSTANCE', 'CHEMICAL_BNF_NAME', 'BNF_CHEMICAL_SUBSTANCE_NAME')
    sect_c  = pick(cols, 'BNF_SECTION', 'SECTION')
    items_c = pick(cols, 'ITEMS', 'ITEM_COUNT', 'TOTAL_ITEMS')
    ym_c    = pick(cols, 'YEAR_MONTH', 'YEARMONTH', 'YM')
    if not (code_c and drug_c and items_c):
        raise ValueError(f'{src["name"]}: columns not recognised: {list(cols)[:8]}')
    use = [c for c in [code_c, name_c, drug_c, sect_c, items_c, ym_c] if c]
    df = pd.read_csv(path, usecols=use)
    ren = {code_c:'AREA_CODE', drug_c:'DRUG', items_c:'ITEMS'}
    if name_c: ren[name_c] = 'AREA_NAME'
    if sect_c: ren[sect_c] = 'SECTION'
    if ym_c:   ren[ym_c]   = 'YM'
    df = df.rename(columns=ren)
    if 'AREA_NAME' not in df: df['AREA_NAME'] = df['AREA_CODE']
    if 'SECTION'  not in df: df['SECTION'] = ''
    df['ITEMS'] = pd.to_numeric(df['ITEMS'], errors='coerce').fillna(0)
    full_parts.append(df.groupby(['AREA_CODE','AREA_NAME','DRUG','SECTION'], as_index=False)['ITEMS'].sum())
    if 'YM' in df:
        df['YEAR'] = df['YM'].astype(str).str[:4]
        year_parts.append(df.groupby(['AREA_CODE','DRUG','YEAR'], as_index=False)['ITEMS'].sum())
    if tmp: os.remove(path)
    print(f'[{i}/{len(SOURCES)}] {src["name"]}  rows={len(df):,}  code={code_c} name={name_c} drug={drug_c}')

FULL = pd.concat(full_parts).groupby(['AREA_CODE','AREA_NAME','DRUG','SECTION'], as_index=False)['ITEMS'].sum()
YEARLY = (pd.concat(year_parts).groupby(['AREA_CODE','DRUG','YEAR'], as_index=False)['ITEMS'].sum() if year_parts else pd.DataFrame())
print('\nFULL  :', FULL.shape, '| YEARLY:', YEARLY.shape)
print('area codes:', FULL['AREA_CODE'].nunique(), '| area names:', FULL['AREA_NAME'].nunique(), '| drugs:', FULL['DRUG'].nunique())

### 4. Save the aggregates
Writes the two CSV files to `DRIVE_DIR` and downloads them. Copy them into `data/` of the repository.

In [ ]:
FULL.to_csv(f'{DRIVE_DIR}/icb_area_substance_items_full.csv', index=False)
if len(YEARLY): YEARLY.to_csv(f'{DRIVE_DIR}/icb_substance_items_by_year.csv', index=False)
print('saved to', DRIVE_DIR)
import hashlib
for fn in ['icb_area_substance_items_full.csv', 'icb_substance_items_by_year.csv']:
    print(fn, 'SHA-256', hashlib.sha256(open(f'{DRIVE_DIR}/{fn}', 'rb').read()).hexdigest())
from google.colab import files
FULL.to_csv('icb_area_substance_items_full.csv', index=False); files.download('icb_area_substance_items_full.csv')
if len(YEARLY):
    YEARLY.to_csv('icb_substance_items_by_year.csv', index=False); files.download('icb_substance_items_by_year.csv')

**Check:** the printout should read 42 area codes, 84 area names (each area appears under its STP name and its ICB name) and 2,161 item names (2,160 after removing leading and trailing spaces, as `code/analyze_af.py` does). The checksums printed in step 4 can be compared with those of the files in `data/` (they can differ if NHSBSA has revised the source files or if a different pandas version writes the numbers differently): SHA-256 c28b170ae6ff3c7dad39945c44b11f8fc92d912116fab31879b828e35f037775 (`icb_area_substance_items_full.csv`) and 3056c5d31542f30250f32a035e52c299f71a6eb82a283b7ec160960c670f45a1 (`icb_substance_items_by_year.csv`).

The quarterly totals in `data/temporal_data.csv` (items, net ingredient cost, number of chemical substances) are made from the same monthly files by `code/quarterly_totals.py`.